In [2]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
df =  pd.read_parquet("all_teams_last10seasons_with_opponent_rolls.parquet")

target = 'TARGET_WL'
exclude_cols = [
    'Team_ID',      # Note: case-sensitive
    'Team_ID_opp',  # Also exclude opponent ID
    'Game_ID',      # Note: case-sensitive
    target
]

# Create features list by excluding the right columns
features = [col for col in df.columns if col not in exclude_cols]


df_model = df.dropna(subset=features + [target])

# Split chronologically (e.g., 80% train, 20% test)
split_idx = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_idx]
test_df = df_model.iloc[split_idx:]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

bool_cols = X_train.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X_train[bool_cols] = X_train[bool_cols].astype(int)
    X_test[bool_cols] = X_test[bool_cols].astype(int)
    
y_train = y_train.astype(int)
y_test = y_test.astype(int)

/var/folders/14/drwtsk3n2wvdsmw2rj7tqh040000gn/T/ipykernel_49606/2252778110.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[bool_cols] = X_train[bool_cols].astype(int)
/var/folders/14/drwtsk3n2wvdsmw2rj7tqh040000gn/T/ipykernel_49606/2252778110.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[bool_cols] = X_test[bool_cols].astype(int)


In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000))
])

param_grid = {
    'logreg__penalty': ['l1', 'l2', 'elasticnet'],
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__solver': ['liblinear', 'saga'],  # saga supports elasticnet
    'logreg__l1_ratio': [0, 0.25, 0.5, 0.75, 1]  # only used for elasticnet
}

grid_lr = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train, y_train)

print("Best Params:", grid_lr.best_params_)
print("Best CV Score:", round(grid_lr.best_score_, 3))

y_pred_lr = grid_lr.predict(X_test)
print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred_lr), 3))
print(classification_report(y_test, y_pred_lr))


/Users/joshuademontigny/Downloads/ML_NBA_Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/joshuademontigny/Downloads/ML_NBA_Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/joshuademontigny/Downloads/ML_NBA_Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/joshuademontigny/Downloads/ML_NBA_Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/joshuademontigny/Downloads/ML_NBA_Project/.venv/lib/python3.1

KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced']
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42),
                       param_grid_rf,
                       cv=5,
                       scoring='accuracy',
                       n_jobs=-1)

grid_rf.fit(X_train, y_train)

print("Best Params:", grid_rf.best_params_)
print("Best CV Score:", round(grid_rf.best_score_, 3))

y_pred_rf = grid_rf.predict(X_test)
print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred_rf), 3))
print(classification_report(y_test, y_pred_rf))

Best Params: {'class_weight': None, 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Best CV Score: 0.607

Test Accuracy: 0.634
              precision    recall  f1-score   support

           0       0.71      0.67      0.69      1206
           1       0.53      0.58      0.55       777

    accuracy                           0.63      1983
   macro avg       0.62      0.62      0.62      1983
weighted avg       0.64      0.63      0.64      1983



In [ ]:
from sklearn.svm import SVC

pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

param_grid_svm = {
    'svm__C': [0.1, 1, 10, 50, 100],
    'svm__gamma': ['scale', 0.1, 0.01, 0.001],
    'svm__kernel': ['rbf', 'poly'],
}

grid_svm = GridSearchCV(pipe_svm, param_grid_svm,
                         cv=5, scoring='accuracy', n_jobs=-1)

grid_svm.fit(X_train, y_train)

print("Best Params:", grid_svm.best_params_)
print("Best CV Score:", round(grid_svm.best_score_, 3))

y_pred_svm = grid_svm.predict(X_test)
print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred_svm), 3))
print(classification_report(y_test, y_pred_svm))


Best Params: {'svm__C': 1, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Best CV Score: 0.601

Test Accuracy: 0.634
              precision    recall  f1-score   support

           0       0.71      0.68      0.69      1206
           1       0.53      0.56      0.55       777

    accuracy                           0.63      1983
   macro avg       0.62      0.62      0.62      1983
weighted avg       0.64      0.63      0.64      1983



In [ ]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced']
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42),
                       param_grid_rf,
                       cv=5,
                       scoring='accuracy',
                       n_jobs=-1)

grid_rf.fit(X_train, y_train)

print("Best Params:", grid_rf.best_params_)
print("Best CV Score:", round(grid_rf.best_score_, 3))

y_pred_rf = grid_rf.predict(X_test)
print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred_rf), 3))
print(classification_report(y_test, y_pred_rf))

Best Params: {'class_weight': None, 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Best CV Score: 0.607

Test Accuracy: 0.634
              precision    recall  f1-score   support

           0       0.71      0.67      0.69      1206
           1       0.53      0.58      0.55       777

    accuracy                           0.63      1983
   macro avg       0.62      0.62      0.62      1983
weighted avg       0.64      0.63      0.64      1983



In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# --- Define parameter grid ---
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(
    objective="binary:logistic",  # <--- binary
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)


# --- Set up GridSearchCV ---
grid_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# --- Fit to training data ---
grid_xgb.fit(X_train, y_train)

# --- Best parameters and CV score ---
print("Best Params:", grid_xgb.best_params_)
print("Best CV Score:", round(grid_xgb.best_score_, 3))

# --- Predictions on test set ---
y_pred_xgb = grid_xgb.predict(X_test)
print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred_xgb), 3))
print(classification_report(y_test, y_pred_xgb))